In [1]:
import cv2

cap = cv2.VideoCapture(4)

if not cap.isOpened():
    print("카메라 열기 실패")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        print("프레임 못 받음")
        break

    cv2.imshow("D435i Camera", frame)

    if cv2.waitKey(1) & 0xFF == 27:  # ESC 종료
        break

cap.release()
cv2.destroyAllWindows()

QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvidia/rby1/rby1_venv/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/nvid

KeyboardInterrupt: 

# Streaming [Black Box Detection]

In [ ]:
import cv2
import numpy as np
import pyrealsense2 as rs


def find_block_center(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # 흰 판 영역 찾기
    # Hard condition
    # lower_white = np.array([0, 0, 180])
    # upper_white = np.array([180, 60, 255])
    # Soft condition
    lower_white = np.array([0, 0, 140])
    upper_white = np.array([180, 90, 255])
    white_mask = cv2.inRange(hsv, lower_white, upper_white)

    # kernel = np.ones((7, 7), np.uint8)
    kernel = np.ones((5, 5), np.uint8)
    white_mask = cv2.morphologyEx(white_mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours( 
        white_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        return None, None

    table = max(contours, key=cv2.contourArea)

    table_mask = np.zeros_like(white_mask)
    cv2.drawContours(table_mask, [table], -1, 255, -1)

    # 흰 판 안의 검은 블록 찾기
    lower_black = np.array([0, 0, 0])
    upper_black = np.array([180, 255, 70])
    black_mask = cv2.inRange(hsv, lower_black, upper_black)

    block_mask = cv2.bitwise_and(black_mask, table_mask)
    block_mask = cv2.morphologyEx(block_mask, cv2.MORPH_OPEN, kernel)

    contours, _ = cv2.findContours(
        block_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    candidates = []

    for cnt in contours:
        area = cv2.contourArea(cnt)

        if 300 < area < 20000:
            candidates.append(cnt)

    if not candidates:
        return None, block_mask

    block = max(candidates, key=cv2.contourArea)

    M = cv2.moments(block)
    if M["m00"] == 0:
        return None, block_mask

    cx = int(M["m10"] / M["m00"])
    cy = int(M["m01"] / M["m00"])

    return (cx, cy, block), block_mask


# RealSense 설정
pipeline = rs.pipeline()
config = rs.config()

config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)
config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)

profile = pipeline.start(config)

# depth를 color 이미지에 정렬
align = rs.align(rs.stream.color)

try:
    while True:
        frames = pipeline.wait_for_frames()
        frames = align.process(frames)

        color_frame = frames.get_color_frame()
        depth_frame = frames.get_depth_frame()

        if not color_frame or not depth_frame:
            continue

        frame = np.asanyarray(color_frame.get_data())

        result, mask = find_block_center(frame)

        if result is not None:
            cx, cy, block = result

            # 중심 픽셀의 depth 값, 단위 meter
            z = depth_frame.get_distance(cx, cy)

            if z > 0:
                depth_intrin = depth_frame.profile.as_video_stream_profile().intrinsics

                X, Y, Z = rs.rs2_deproject_pixel_to_point(
                    depth_intrin,
                    [cx, cy],
                    z
                )

                print(
                    f"pixel=({cx}, {cy}) | "
                    f"X={X:.3f}m, Y={Y:.3f}m, Z={Z:.3f}m"
                )

                cv2.drawContours(frame, [block], -1, (0, 255, 0), 2)
                cv2.circle(frame, (cx, cy), 6, (0, 0, 255), -1)

                cv2.putText(
                    frame,
                    f"X:{X:.2f} Y:{Y:.2f} Z:{Z:.2f} m",
                    (cx + 10, cy),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (0, 0, 255),
                    2
                )

        cv2.imshow("D435i Block Detection", frame)

        if mask is not None:
            cv2.imshow("mask", mask)

        # ESC 누르면 종료
        if cv2.waitKey(1) & 0xFF == 27:
            break

finally:
    pipeline.stop()
    cv2.destroyAllWindows()

pixel=(235, 372) | X=-0.099m, Y=0.168m, Z=0.812m
pixel=(235, 372) | X=-0.092m, Y=0.156m, Z=0.755m
pixel=(235, 372) | X=-0.092m, Y=0.156m, Z=0.754m
pixel=(235, 372) | X=-0.092m, Y=0.157m, Z=0.757m
pixel=(234, 372) | X=-0.097m, Y=0.163m, Z=0.788m
pixel=(234, 372) | X=-0.097m, Y=0.163m, Z=0.788m
pixel=(234, 371) | X=-0.097m, Y=0.161m, Z=0.786m
pixel=(234, 371) | X=-0.095m, Y=0.159m, Z=0.773m
pixel=(234, 371) | X=-0.101m, Y=0.168m, Z=0.821m
pixel=(234, 371) | X=-0.094m, Y=0.156m, Z=0.758m
pixel=(234, 372) | X=-0.096m, Y=0.162m, Z=0.782m
pixel=(234, 372) | X=-0.099m, Y=0.166m, Z=0.801m
pixel=(234, 372) | X=-0.091m, Y=0.153m, Z=0.739m
pixel=(234, 372) | X=-0.093m, Y=0.156m, Z=0.754m
pixel=(234, 372) | X=-0.094m, Y=0.158m, Z=0.763m
pixel=(234, 372) | X=-0.094m, Y=0.157m, Z=0.760m
pixel=(234, 372) | X=-0.100m, Y=0.167m, Z=0.809m
pixel=(234, 372) | X=-0.098m, Y=0.164m, Z=0.794m
pixel=(234, 372) | X=-0.099m, Y=0.165m, Z=0.799m
pixel=(234, 372) | X=-0.097m, Y=0.163m, Z=0.789m
pixel=(234, 372) | X

KeyboardInterrupt: 

# [Black Box Detection] 5 seconds Average

In [10]:
import cv2
import numpy as np
import pyrealsense2 as rs
import time


def find_block_center(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    lower_white = np.array([0, 0, 180])
    upper_white = np.array([180, 60, 255])
    white_mask = cv2.inRange(hsv, lower_white, upper_white)

    kernel = np.ones((7, 7), np.uint8)
    white_mask = cv2.morphologyEx(white_mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(
        white_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours:
        return None, None

    table = max(contours, key=cv2.contourArea)

    table_mask = np.zeros_like(white_mask)
    cv2.drawContours(table_mask, [table], -1, 255, -1)

    lower_black = np.array([0, 0, 0])
    upper_black = np.array([180, 255, 70])
    black_mask = cv2.inRange(hsv, lower_black, upper_black)

    block_mask = cv2.bitwise_and(black_mask, table_mask)
    block_mask = cv2.morphologyEx(block_mask, cv2.MORPH_OPEN, kernel)

    contours, _ = cv2.findContours(
        block_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    candidates = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if 300 < area < 20000:
            candidates.append(cnt)

    if not candidates:
        return None, block_mask

    block = max(candidates, key=cv2.contourArea)

    M = cv2.moments(block)
    if M["m00"] == 0:
        return None, block_mask

    cx = int(M["m10"] / M["m00"])
    cy = int(M["m01"] / M["m00"])

    return (cx, cy, block), block_mask


def get_block_position_5sec():
    pipeline = rs.pipeline()
    config = rs.config()

    config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)
    config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)

    pipeline.start(config)
    align = rs.align(rs.stream.color)

    X_list, Y_list, Z_list = [], [], []

    start_time = time.time()

    try:
        while time.time() - start_time < 5:  # 5초 동안만 실행
            frames = pipeline.wait_for_frames()
            frames = align.process(frames)

            color_frame = frames.get_color_frame()
            depth_frame = frames.get_depth_frame()

            if not color_frame or not depth_frame:
                continue

            frame = np.asanyarray(color_frame.get_data())

            result, _ = find_block_center(frame)

            if result is None:
                continue

            cx, cy, _ = result
            z = depth_frame.get_distance(cx, cy)

            if z <= 0:
                continue

            depth_intrin = depth_frame.profile.as_video_stream_profile().intrinsics

            X, Y, Z = rs.rs2_deproject_pixel_to_point(
                depth_intrin,
                [cx, cy],
                z
            )

            print(f"X={X:.3f}, Y={Y:.3f}, Z={Z:.3f}")

            X_list.append(X)
            Y_list.append(Y)
            Z_list.append(Z)

    finally:
        pipeline.stop()

    # 평균 계산
    if len(X_list) == 0:
        print("값을 얻지 못함")
        return None

    X_avg = np.mean(X_list)
    Y_avg = np.mean(Y_list)
    Z_avg = np.mean(Z_list)

    print("\n===== 평균값 =====")
    print(f"X={X_avg:.3f}, Y={Y_avg:.3f}, Z={Z_avg:.3f}")

    return X_avg, Y_avg, Z_avg


# 실행
pos = get_block_position_5sec()
print("최종 결과:", pos)

X=-0.083, Y=0.037, Z=1.049
X=-0.083, Y=0.037, Z=1.048
X=-0.082, Y=0.037, Z=1.037
X=-0.080, Y=0.036, Z=1.013
X=-0.081, Y=0.037, Z=1.032
X=-0.081, Y=0.037, Z=1.025
X=-0.080, Y=0.036, Z=1.016
X=-0.081, Y=0.037, Z=1.030
X=-0.083, Y=0.038, Z=1.053
X=-0.082, Y=0.037, Z=1.037
X=-0.081, Y=0.036, Z=1.020
X=-0.081, Y=0.036, Z=1.020
X=-0.082, Y=0.037, Z=1.039
X=-0.080, Y=0.036, Z=1.018
X=-0.083, Y=0.037, Z=1.051
X=-0.081, Y=0.036, Z=1.021
X=-0.080, Y=0.036, Z=1.015
X=-0.083, Y=0.037, Z=1.049
X=-0.080, Y=0.036, Z=1.015
X=-0.081, Y=0.036, Z=1.020
X=-0.081, Y=0.036, Z=1.023
X=-0.082, Y=0.037, Z=1.039
X=-0.082, Y=0.037, Z=1.040
X=-0.082, Y=0.037, Z=1.035
X=-0.082, Y=0.035, Z=1.039
X=-0.084, Y=0.038, Z=1.062
X=-0.081, Y=0.037, Z=1.030
X=-0.080, Y=0.036, Z=1.010
X=-0.082, Y=0.037, Z=1.040
X=-0.081, Y=0.037, Z=1.032
X=-0.082, Y=0.037, Z=1.033
X=-0.080, Y=0.036, Z=1.011
X=-0.083, Y=0.038, Z=1.057
X=-0.083, Y=0.038, Z=1.055
X=-0.083, Y=0.038, Z=1.057
X=-0.081, Y=0.035, Z=1.027
X=-0.081, Y=0.035, Z=1.030
X